# Build a vector store from the DispatchDesk corpus

Load every operational Markdown document in `data/corpus`, split it by its `##` sections, embed each chunk, and persist the resulting Chroma vector store locally. Run this notebook from the repository root.

Install the notebook dependencies first: `pip install langchain langchain-community langchain-chroma langchain-huggingface sentence-transformers` (or add them to the project's dependency configuration). The first embedding run downloads the selected model.

In [ ]:
%pip install -q langchain langchain-groq faiss-cpu gradio pypdf sentence-transformers python-dotenv langchain-community

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain-0.3.30-py3-none-any.whl (1.0 MB)
  Using cached langchain_community-0.3.31-py3-none-any.whl (2.5 MB)
  Using cached sentence_transformers-5.1.2-py3-none-any.whl (488 kB)
  Using cached langchain_core-0.3.86-py3-none-any.whl (461 kB)
  Using cached sqlalchemy-2.0.54-cp39-cp39-macosx_11_0_arm64.whl (2.2 MB)
  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
  Using cached langsmith-0.4.37-py3-none-any.whl (396 kB)
  Using cached pyyaml-6.0.3-cp39-cp39-macosx_11_0_arm64.whl (174 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl (33 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl (472 kB)
  Using cached aiohttp-3.13.5-cp39-cp39-macosx_11_0_arm64.whl (499 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl (28 kB)
  Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl (5.3 MB)
  Using cached httpx_ss

In [13]:
from pathlib import Path
import re

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

REPO_ROOT = Path.cwd()
REPO_ROOT = Path("/Users/ravi/Projects/supply-chain-bv")
CORPUS_DIR = REPO_ROOT / "data" / "corpus"
VECTORSTORE_DIR = REPO_ROOT / "data" / "vectorstore"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

if not CORPUS_DIR.is_dir():
    raise FileNotFoundError(f"Corpus directory not found: {CORPUS_DIR}. Run this notebook from the repository root.")

print(f"Corpus: {CORPUS_DIR}")
print(f"Vector store: {VECTORSTORE_DIR}")

Corpus: /Users/ravi/Projects/supply-chain-bv/data/corpus
Vector store: /Users/ravi/Projects/supply-chain-bv/data/vectorstore


## Load and chunk source documents

The corpus README is documentation, so it is excluded from operational retrieval. Each document is split on level-two headings to keep policy sections together. Each chunk retains the corpus `doc_id`, title, version, section, and relative source path. The first chunk also includes the document's introductory text before the first section heading.

In [14]:
def parse_metadata(markdown: str, source: Path) -> dict[str, str]:
    title_match = re.search(r"(?m)^#\s+(.+?)\s*$", markdown)
    doc_id_match = re.search(r"(?im)^\*\*Document ID:\*\*\s*(.+?)\s*$", markdown)
    version_match = re.search(r"(?im)^\*\*Version:\*\*\s*(.+?)\s*$", markdown)
    return {
        "doc_id": doc_id_match.group(1).strip() if doc_id_match else source.stem,
        "title": title_match.group(1).strip() if title_match else source.stem,
        "version": version_match.group(1).strip() if version_match else "unknown",
        "source": source.relative_to(REPO_ROOT).as_posix(),
    }


def section_chunks(markdown: str) -> list[tuple[str, str]]:
    """Return (section heading, section text) pairs, split at ## headings."""
    matches = list(re.finditer(r"(?m)^##\s+(.+?)\s*$", markdown))
    if not matches:
        return [("Document", markdown.strip())] if markdown.strip() else []

    sections: list[tuple[str, str]] = []
    intro = markdown[:matches[0].start()].strip()
    if intro:
        sections.append(("Introduction", intro))

    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(markdown)
        heading = match.group(1).strip()
        body = markdown[match.start():end].strip()
        if body:
            sections.append((heading, body))
    return sections


source_files = sorted(path for path in CORPUS_DIR.glob("*.md") if path.name.lower() != "readme.md")
if not source_files:
    raise FileNotFoundError(f"No operational Markdown documents found in {CORPUS_DIR}")

documents: list[Document] = []
for source in source_files:
    markdown = source.read_text(encoding="utf-8")
    metadata = parse_metadata(markdown, source)
    for section, content in section_chunks(markdown):
        documents.append(Document(page_content=content, metadata={**metadata, "section": section}))

print(f"Loaded {len(source_files)} documents into {len(documents)} section chunks")
for document in documents:
    print(f"{document.metadata['doc_id']} | {document.metadata['section']}")

Loaded 6 documents into 36 section chunks
DD-SOP-001 | Introduction
DD-SOP-001 | Purpose and source boundaries
DD-SOP-001 | Triage sequence
DD-SOP-001 | Draft-only actions and approvals
DD-SOP-001 | Missing data and uncertainty
DD-SOP-001 | Memory and preference handling
DD-DIAG-001 | Introduction
DD-DIAG-001 | Delivery stages and diagnostic reference points
DD-DIAG-001 | Diagnosis from historical metrics
DD-DIAG-001 | Diagnosis from a live queue
DD-DIAG-001 | Rain and surge interpretation
DD-DIAG-001 | Response pattern
DD-DIAG-001 | Worked-method example (method only)
DD-BATCH-001 | Introduction
DD-BATCH-001 | Eligibility is conjunctive
DD-BATCH-001 | Required inputs before recommending a specific pair
DD-BATCH-001 | How to present a batch proposal
DD-BATCH-001 | Preference interaction
DD-BATCH-001 | No ETA inference from batching
DD-WEATHER-001 | Introduction
DD-WEATHER-001 | Diagnose before choosing a lever
DD-WEATHER-001 | Safe operating sequence
DD-WEATHER-001 | Declared surge and

## Embed and persist the vector store

This writes a persistent local Chroma database under `data/vectorstore`. Re-running the build clears the existing collection first, so source edits are reflected without leaving stale chunks.

In [15]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
)

VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)
vectorstore = Chroma(
    collection_name="dispatchdesk_corpus",
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)

# Rebuild deterministically from the current corpus.
existing = vectorstore.get(include=[])
if existing["ids"]:
    vectorstore.delete(ids=existing["ids"])

vectorstore.add_documents(documents)
print(f"Stored {len(documents)} embedded chunks in {VECTORSTORE_DIR}")

Stored 36 embedded chunks in /Users/ravi/Projects/supply-chain-bv/data/vectorstore


## Check retrieval

Use this cell to confirm that the persisted store returns relevant sections and their source metadata.

In [16]:
query = "When should two orders be considered for batching, and what cold-chain limits apply?"
for result in vectorstore.similarity_search_with_relevance_scores(query, k=4):
    document, score = result
    print(f"score={score:.3f} | {document.metadata}")
    print(document.page_content[:700], "\n")

score=0.509 | {'doc_id': 'DD-BATCH-001', 'source': 'data/corpus/03-batching-and-cold-chain.md', 'title': 'Order batching and cold-chain rules', 'version': '1.0 (2026-09-28)', 'section': 'Introduction'}
# Order batching and cold-chain rules

**Document ID:** DD-BATCH-001  
**Version:** 1.0 (2026-09-28)  
**Authority:** Simulated demo rules explicitly listed in `docs/6-pager.md` Appendix D and `docs/initial/requirements.md`.  
**Applies to:** Any suggestion to place more than one order on a rider's trip. 

score=0.396 | {'source': 'data/corpus/03-batching-and-cold-chain.md', 'title': 'Order batching and cold-chain rules', 'section': 'Eligibility is conjunctive', 'version': '1.0 (2026-09-28)', 'doc_id': 'DD-BATCH-001'}
## Eligibility is conjunctive

A batch is eligible only when every applicable check below passes using the order and routing attributes returned by tools. A manager approval or incentive does not waive cold-chain, geography, detour, order-age, or safety rules.

1. **Order c

In [ ]:
from langchain_groq import ChatGroq
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.text_splitter import CharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from getpass import getpass
import gradio as gr
import os

os.environ["GROQ_API_KEY"] = getpass("🔑 Enter your Groq API key: ")


In [21]:
llm = ChatGroq(model_name="openai/gpt-oss-20b")

prompt_template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Answer:"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    chain_type="stuff",
    chain_type_kwargs={"prompt": PROMPT}
)

def chat_with_memory(message, history):
    response = qa_chain.run(message)
    return response


gr.ChatInterface(
    fn=chat_with_memory,
    title="Dispatch Desk RAG"
).launch(share=True)

Running on local URL:  http://127.0.0.1:7862
Running on public URL: https://a50a787b5fc17c3d6f.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
